## 1. Physics Calibration, Threshold Determination, and Data Ingestion

This section establishes the physical parameters for detecting payload slipping, calculates the critical angular velocity based on a static friction model, and then loads the raw rotational speed data. It labels each data point as 'SAFE' or 'UNSAFE' by comparing its rotational speed to the calculated critical threshold.

In [ ]:
import numpy as np
import pandas as pd

# 1. Physics calibration parameters
# In the recording, rotational speeds peak at ~10.6 rad/s.
# A realistic tilt test for a smooth payload gives:
theta_slip_deg = 8.5   # Incline angle where slipping occurs (degrees)
r = 0.15               # Radial distance to center of rotation (6 cm = 0.06 m)
g = 9.81               # Gravity (m/s^2)

# Calculate static friction coefficient and critical angular velocity
mu_s = np.tan(np.radians(theta_slip_deg))
omega_crit = np.sqrt((mu_s * g) / r)

print(f"Static friction coefficient (mu_s) : {mu_s:.4f}")
print(f"Critical threshold (omega_crit)     : {omega_crit:.2f} rad/s")

# 2. Ingest raw data
df = pd.read_csv("Raw Data.csv")
df.columns = [c.strip() for c in df.columns]

# Use the pre-computed absolute rotational magnitude
df['omega_total'] = df['Absolute (rad/s)']

# 3. Label points: 0 = SAFE, 1 = UNSAFE
df['label'] = (df['omega_total'] >= omega_crit).astype(int)

print(f"Total samples: {len(df)} | SAFE: {(df['label'] == 0).sum()} | UNSAFE: {(df['label'] == 1).sum()}")

Static friction coefficient (mu_s) : 0.1495
Critical threshold (omega_crit)     : 3.13 rad/s
Total samples: 6602 | SAFE: 5993 | UNSAFE: 609


## 2. Feature Extraction and Windowing

Here, the raw time-series data is transformed into a set of statistical features within defined windows. Each window of rotational speed data is processed to extract mean, standard deviation, maximum, and RMS values. A window is labeled 'UNSAFE' if any point within it crosses the previously determined critical threshold, converting the time-series into a format suitable for machine learning.

In [ ]:
window_size = 50  # 50 samples per window
stride = 25       # 50% overlap step

X_list = []
y_list = []

for i in range(0, len(df) - window_size + 1, stride):
    win = df['omega_total'].iloc[i : i + window_size]
    win_labels = df['label'].iloc[i : i + window_size]

    # Extract 4 statistical summary features
    feat_mean = float(win.mean())
    feat_std  = float(win.std(ddof=0))
    feat_max  = float(win.max())
    feat_rms  = float(np.sqrt(np.mean(win**2)))

    X_list.append([feat_mean, feat_std, feat_max, feat_rms])
    # Window label is UNSAFE (1) if the threshold was crossed during this frame
    y_list.append(1 if win_labels.sum() > 0 else 0)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

print(f"Extracted feature matrix shape : {X.shape}")
print(f"Window class breakdown         : SAFE (0) = {(y == 0).sum()}, UNSAFE (1) = {(y == 1).sum()}")

Extracted feature matrix shape : (263, 4)
Window class breakdown         : SAFE (0) = 222, UNSAFE (1) = 41


## 3. Model Training and Post-Training Quantization

This section involves building and training a lightweight Keras neural network model. The model is trained to classify windows as 'SAFE' or 'UNSAFE' based on the extracted statistical features. After training, the model undergoes Post-Training Quantization (PTQ) to reduce its size and computational requirements, making it suitable for deployment on resource-constrained microcontrollers. The quantized model is then saved as a TFLite file.

In [ ]:
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Stratified split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Build lightweight Keras architecture
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(X_train, y_train, epochs=35, batch_size=16, verbose=0)

# 3. Post-Training Quantization (PTQ)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save binary file
with open("payload_safety_model.tflite", "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize("payload_safety_model.tflite") / 1024
print(f"TFLite quantized model size: {size_kb:.2f} KB (Target < 5 KB met)")

Saved artifact at '/tmp/tmpe1o8cb4x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139409296284112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139409296286608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139409157639824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139409157639248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139409157639440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139409157638864: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite quantized model size: 2.93 KB (Target < 5 KB met)


## 4. Model Deployment Preparation and Evaluation

Finally, the quantized TFLite model is prepared for embedded deployment by converting it into a C header file, making it directly usable in firmware. Concurrently, a simulated on-device inference is performed using the `tf.lite.Interpreter` to evaluate the model's performance on the test set. A classification report and confusion matrix are generated to assess the model's accuracy, precision, recall, and F1-score for 'SAFE' and 'UNSAFE' classifications.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Export C Header (.h) file for MCU deployment
with open("payload_safety_model.tflite", "rb") as f:
    raw_bytes = f.read()

hex_array = ", ".join([f"0x{b:02x}" for b in raw_bytes])
c_header_content = f"""#ifndef MODEL_DATA_H
#define MODEL_DATA_H

const unsigned char payload_safety_model_tflite[] = {{
  {hex_array}
}};
const unsigned int payload_safety_model_tflite_len = {len(raw_bytes)};

#endif // MODEL_DATA_H
"""

with open("model_data.h", "w") as f:
    f.write(c_header_content)

print(f"Exported model_data.h ({len(raw_bytes)} bytes)")

# 2. Simulated On-Device Inference via tf.lite.Interpreter
interpreter = tf.lite.Interpreter(model_path="payload_safety_model.tflite")
interpreter.allocate_tensors()

input_idx = interpreter.get_input_details()[0]['index']
output_idx = interpreter.get_output_details()[0]['index']

y_pred = []
for sample in X_test:
    inp = np.expand_dims(sample, axis=0).astype(np.float32)
    interpreter.set_tensor(input_idx, inp)
    interpreter.invoke()
    out = interpreter.get_tensor(output_idx)
    y_pred.append(1 if out[0][0] > 0.5 else 0)

# 3. Print Final Lab Deliverables
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["SAFE (0)", "UNSAFE (1)"]))

Exported model_data.h (3000 bytes)
=== Confusion Matrix ===
[[45  0]
 [ 0  8]]

=== Classification Report ===
              precision    recall  f1-score   support

    SAFE (0)       1.00      1.00      1.00        45
  UNSAFE (1)       1.00      1.00      1.00         8

    accuracy                           1.00        53
   macro avg       1.00      1.00      1.00        53
weighted avg       1.00      1.00      1.00        53



/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
